# TP12: Sistemas Embebidos - Plataforma Arduino Uno

**Asignatura:** Ingeniería de Software II  
**Docente:** Dr. Pedro E. Colla  
**Institución:** Facultad de Ciencia y Tecnología (FCyT) - Universidad Autónoma de Entre Ríos (UADER)  
**Alumno:** Jesús Arévalo  
**Ubicación de Código y Firmware:** `src/tp12/`  

> **Resumen del Práctico:** En el presente trabajo se diseña, implementa, audita y valida el firmware embebido y el circuito de control para una placa **Arduino Uno (microcontrolador ATmega328P)**. El sistema integra un pulsador físico (entrada digital D2), un potenciómetro lineal de 10 kΩ (entrada analógica A0) y el LED onboard (pin 13). Se corrigen las anomalías del sketch base provisto por la cátedra (`is2_taller_sistemas_embebidos.ino`) y se implementan las modificaciones solicitadas: ritmo de parpadeo de 5 segundos (a), parpadeo de 1 segundo (b) y ajuste del umbral de activación al 25% de la tensión de la fuente (c).

---

## 1. Fundamentos de Arquitectura Embebida y Conversión A/D

### 1.1. Conversor Analógico-Digital (ADC) de 10 bits (ATmega328P)
El microcontrolador ATmega328P posee un conversor A/D de aproximaciones sucesivas con resolución de $N = 10$ bits, lo que define $2^{10} = 1024$ niveles de cuantización discretos (rango entero de $0$ a $1023$). Con una tensión de referencia de $V_{ref} = 5.0\text{ V}$ provista por el regulador de la placa:

$$\text{Resolución (LSB)} = \Delta V = \frac{V_{ref}}{2^{10} - 1} = \frac{5.0\text{ V}}{1023} \approx 4.8876\text{ mV/cuentas}$$

La relación matemática entre la tensión continua medida en el cursor del potenciómetro $V_{in} \in [0, 5.0]\text{ V}$ y la lectura devuelta por `analogRead(A0)` es:

$$\text{ADC}_{val} = \left\lfloor \frac{V_{in}}{V_{ref}} \times 1023 \right\rfloor$$

$$\hat{V}_{in} = \frac{\text{ADC}_{val} \times V_{ref}}{1023}$$

### 1.2. Umbrales de Comparación del TP12
- **Consigna Base / Modificaciones (a) y (b) (50% de la tensión de alimentación):**
  $$V_{th} = 0.50 \times 5.0\text{ V} = 2.50\text{ V} \implies \text{ADC}_{th} = \lfloor 0.50 \times 1023 \rfloor = 511\text{ cuentas}$$
- **Modificación (c) (25% de la tensión de alimentación):**
  $$V_{th} = 0.25 \times 5.0\text{ V} = 1.25\text{ V} \implies \text{ADC}_{th} = \lfloor 0.25 \times 1023 \rfloor = 255\text{ cuentas}$$

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulador_embebido import CircuitConfig, EmuladorArduinoTP12, voltage_to_adc

# Configuración base: 50% de umbral, 1s de parpadeo
config_base = CircuitConfig(umbral_ratio=0.50, periodo_parpadeo_ms=1000.0)
emulador_base = EmuladorArduinoTP12(config_base)

# Configuración Mod C: 25% de umbral
config_c = CircuitConfig(umbral_ratio=0.25, periodo_parpadeo_ms=1000.0)
emulador_c = EmuladorArduinoTP12(config_c)

print(f'LSB Resolution: {config_base.lsb_voltage * 1000:.3f} mV')
print(f'Umbral Base (50%): {config_base.umbral_adc} cuentas ({config_base.umbral_voltage:.2f} V)')
print(f'Umbral Mod C (25%): {config_c.umbral_adc} cuentas ({config_c.umbral_voltage:.2f} V)')


In [ ]:
# Evaluación de una grilla de tensiones analógicas bajo ambos perfiles
tensiones_test = [0.0, 0.5, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
filas = []

for v in tensiones_test:
    res_base = emulador_base.evaluar_ciclo(button_pressed=True, pot_voltage=v)
    res_c = emulador_c.evaluar_ciclo(button_pressed=True, pot_voltage=v)
    filas.append({
        'Tension_In (V)': v,
        'ADC (10 bits)': res_base['adc_val'],
        'Modo Base (50%) Activado': res_base['activado'],
        'Modo Base LED': res_base['led_modo'],
        'Mod C (25%) Activado': res_c['activado'],
        'Mod C LED': res_c['led_modo'],
    })

df_test = pd.DataFrame(filas)
df_test


In [ ]:
# Graficación de la curva de transferencia y regiones de activación
v_sweep = np.linspace(0, 5, 200)
adc_sweep = [voltage_to_adc(v) for v in v_sweep]

plt.figure(figsize=(9, 5))
plt.plot(v_sweep, adc_sweep, label='Respuesta ADC Cuantizada', color='#1f77b4', lw=2)
plt.axvline(2.5, color='green', linestyle='--', label='Umbral 50% (2.50V, ADC=511)')
plt.axvline(1.25, color='red', linestyle='--', label='Umbral 25% (1.25V, ADC=255)')
plt.axvspan(0, 1.25, color='red', alpha=0.15, label='Zona Activa (Mod C)')
plt.axvspan(1.25, 2.50, color='green', alpha=0.15, label='Zona Activa Adicional (Mod Base)')
plt.title('Curva de Transferencia ADC y Zonas de Activación de LED')
plt.xlabel('Tensión del Potenciómetro [V]')
plt.ylabel('Cuentas ADC (0-1023)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cronogramas de parpadeo temporal: Mod A (5s) vs Mod B (1s)
cfg_mod_a = CircuitConfig(periodo_parpadeo_ms=5000.0)
cfg_mod_b = CircuitConfig(periodo_parpadeo_ms=1000.0)

emu_a = EmuladorArduinoTP12(cfg_mod_a)
emu_b = EmuladorArduinoTP12(cfg_mod_b)

t_s, onda_a = emu_a.generar_cronograma_parpadeo(duracion_total_s=15.0, paso_ms=10.0)
_, onda_b = emu_b.generar_cronograma_parpadeo(duracion_total_s=15.0, paso_ms=10.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.step(t_s, onda_a, where='post', color='#ff7f0e', lw=2)
ax1.set_title('Modificación (a): Ritmo de Parpadeo cada 5 Segundos (T=5s)')
ax1.set_ylabel('LED (0/1)')
ax1.grid(True, linestyle=':', alpha=0.6)

ax2.step(t_s, onda_b, where='post', color='#2ca02c', lw=2)
ax2.set_title('Modificación (b): Ritmo de Parpadeo cada 1 Segundo (T=1s)')
ax2.set_xlabel('Tiempo [segundos]')
ax2.set_ylabel('LED (0/1)')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 2. Auditoría del Firmware Original y Mejoras Introducidas

Durante la inspección de código del archivo `is2_taller_sistemas_embebidos.ino` original, se detectaron cuatro no conformidades con respecto a la especificación de requerimientos:

1. **Entrada analógica desfasada:** El código original invocaba `analogRead(2)` (correspondiente a A2), cuando el cursor del potenciómetro en el circuito de TinkerCad se encuentra cableado a `A0` (incluso declarado en `setup()` con `pinMode(A0, INPUT)`).
2. **Omisión de la lógica de decisión por umbral:** El código original no comparaba la lectura analógica contra el 50% de la fuente (`ADC <= 511`), sino que encendía el LED de forma incondicional toda vez que se pulsaba el botón.
3. **Instrucción redundante/huérfana:** `digitalWrite(3, HIGH);` en la rutina `setup()`, sin declaración previa de `pinMode(3, OUTPUT)` ni conexión en el circuito.
4. **Polaridad del pulsador:** Parametrización mediante constante booleana para soportar esquemas con resistencia pull-down externa (activo en HIGH) o pull-up (activo en LOW).

Las soluciones implementadas satisfacen íntegramente las especificaciones de diseño, verificación y pruebas de la cátedra.